## Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual [here](https://quantra.quantinsti.com/quantra-notebook).
2. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
3. Before running this notebook on your local PC:
   - You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on **"Run Codes Locally on Your Machine"** in the course.
   - You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or Python modules that might be required to run this notebook.


# From Raw Data to Moving Average

Welcome back! In our last notebook, we successfully connected to the Zerodha WebSocket and got a raw stream of tick data. Now, we'll take the next step: parsing that data to calculate a technical indicator.

This is the bridge between simply receiving data and building a trading strategy.

## Outline

1.  **Recap of the Setup**: Briefly revisit the connection code.
2.  **Introducing State**: Creating variables to store price history.
3.  **Modifying `on_ticks`**: Updating our function to parse data, manage history, and calculate the SMA.
4.  **Running the Live Calculation**: Putting it all together to see the live LTP and its moving average.
5.  **Conclusion & Next Steps**: Summarizing our progress and looking ahead to building trading logic.

### 1. The Code So Far

Let's start with the complete code from our last session. It initializes the connection, defines our three callbacks (`on_ticks`, `on_connect`, `on_close`), assigns them, and starts the connection. We will be modifying this code.

In [1]:
# Import the necessary libraries
from kiteconnect import KiteTicker
import logging

# --- Setup ---
logging.basicConfig(level=logging.INFO) # We'll use INFO level for cleaner output

# --- Your Credentials ---
api_key = "YOUR_API_KEY"
access_token = "YOUR_ACCESS_TOKEN"

# --- Initialise the Ticker ---
kws = KiteTicker(api_key, access_token)

### 2. Storing Price History & Calculating SMA

To calculate a moving average, we need to keep track of the most recent prices. We'll create a list called `ltp_history` to store them. We'll also define our Simple Moving Average (SMA) period.

Now, we'll rewrite our `on_ticks` function with the new logic:
1.  **Parse**: For each incoming tick, extract the Last Traded Price (LTP).
2.  **Store**: Add the new LTP to our `ltp_history` list.
3.  **Maintain**: Keep the list at a fixed size (`SMA_PERIOD`) by removing the oldest price.
4.  **Calculate**: If we have enough data, calculate the SMA and print the result.

In [3]:
# --- Global variables for storing data ---
ltp_history = []
SMA_PERIOD = 10 # We'll calculate a 10-tick moving average

# --- Callback Functions (with modified on_ticks) ---

def on_ticks(ws, ticks):
    """This function is called for every new tick.
    We will parse it, store the price, and calculate the SMA.
    """
    # The 'ticks' variable is a list of tick dictionaries.
    for tick in ticks:
        last_price = tick['last_price']
        
        # 1. Add the new price to our list
        ltp_history.append(last_price)
        
        # 2. Make sure our list doesn't grow forever.
        #    We only want to keep the last SMA_PERIOD prices.
        if len(ltp_history) > SMA_PERIOD:
            # Remove the oldest price (the first one in the list)
            ltp_history.pop(0)

        # 3. Check if we have enough data to calculate the SMA
        if len(ltp_history) == SMA_PERIOD:
            # Calculate the average
            sma = sum(ltp_history) / SMA_PERIOD
            
            # Log the output, formatted to 2 decimal places for the SMA
            logging.info(f"LTP: {last_price} | SMA({SMA_PERIOD}): {sma:.2f}")

def on_connect(ws, response):
    """This function runs once the connection is established."""
    logging.info("Successfully connected! WebSocket connection is now open.")
    instrument_tokens = [738561] # Reliance
    ws.subscribe(instrument_tokens)
    ws.set_mode(ws.MODE_LTP, instrument_tokens)
    logging.info(f"Subscribed to tokens {instrument_tokens} in LTP mode.")

def on_close(ws, code, reason):
    """This function runs when the connection is closed."""
    logging.info(f"Connection closed: {code} - {reason}")
    ws.stop()

### 3. Running the Live Calculation

With our new `on_ticks` function ready, the final step is to assign the callbacks and start the connection, just like before. 

When you run the cell below, at first, nothing will be printed. The script needs to collect 10 ticks to fill our `ltp_history` list. After the 10th tick arrives, you will start seeing the live LTP and its corresponding 10-period SMA printed with every new update.

In [4]:
# --- Assign Callbacks and Connect ---
kws.on_ticks = on_ticks
kws.on_connect = on_connect
kws.on_close = on_close

print("Connecting to WebSocket... (This will run indefinitely. Interrupt the kernel to stop.)")
kws.connect()

INFO:root:Successfully connected! WebSocket connection is now open.
INFO:root:Subscribed to tokens [738561] in LTP mode.


Connecting to WebSocket... (This will run indefinitely. Interrupt the kernel to stop.)


INFO:root:LTP: 1485.1 | SMA(10): 1485.07
INFO:root:LTP: 1485.1 | SMA(10): 1485.06
INFO:root:LTP: 1485.1 | SMA(10): 1485.08
INFO:root:LTP: 1485.1 | SMA(10): 1485.08
INFO:root:LTP: 1485.1 | SMA(10): 1485.10
INFO:root:LTP: 1485.1 | SMA(10): 1485.10
INFO:root:LTP: 1484.8 | SMA(10): 1485.07
INFO:root:LTP: 1484.8 | SMA(10): 1485.04
INFO:root:LTP: 1484.8 | SMA(10): 1485.01
INFO:root:LTP: 1484.8 | SMA(10): 1484.98
INFO:root:LTP: 1484.8 | SMA(10): 1484.95
INFO:root:LTP: 1484.8 | SMA(10): 1484.92
INFO:root:LTP: 1484.8 | SMA(10): 1484.89
INFO:root:LTP: 1485.0 | SMA(10): 1484.88
INFO:root:LTP: 1485.0 | SMA(10): 1484.87
INFO:root:LTP: 1485.2 | SMA(10): 1484.88
INFO:root:LTP: 1485.2 | SMA(10): 1484.92
INFO:root:LTP: 1485.2 | SMA(10): 1484.96
INFO:root:LTP: 1485.2 | SMA(10): 1485.00
INFO:root:LTP: 1485.0 | SMA(10): 1485.02
INFO:root:LTP: 1485.0 | SMA(10): 1485.04
INFO:root:LTP: 1485.0 | SMA(10): 1485.06
INFO:root:LTP: 1485.0 | SMA(10): 1485.08
INFO:root:LTP: 1485.0 | SMA(10): 1485.08
INFO:root:LTP: 1

### 4. Conclusion & Next Steps

In this notebook, we took a huge step forward. We went from just *seeing* raw data to *processing* it in real-time.

**Recap:**
- We learned how to parse the `ticks` list to extract the specific data we need (the LTP).
- We used a Python list to store a moving history of the last 10 prices.
- We calculated a live Simple Moving Average with every new tick.

Now that we can calculate an indicator, the next logical question is: **how do we act on it?**

In our next lesson, we'll build on this script to create simple trading logic. For example: 'If the price crosses above its moving average, that could be a buy signal.' We'll lay the groundwork for turning these calculations into trading decisions.